# Retail Sales and Customer Analysis

What drives this store’s revenue, and how important are repeat customers?

First session: load, understand, and audit the source data. One row is an invoice line; count distinct invoices to count orders.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_FILE = ROOT / "data" / "raw" / "Online Retail.xlsx"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Download Online Retail.xlsx from UCI and place it at {DATA_FILE}")
df = pd.read_excel(DATA_FILE)
df.head()

## Understand the columns

| Column | Meaning |
|---|---|
| InvoiceNo | Invoice identifier; invoices contain multiple lines |
| StockCode | Product identifier |
| Description | Product description |
| Quantity | Units on this invoice line |
| InvoiceDate | Transaction date and time |
| UnitPrice | Price per unit |
| CustomerID | Customer identifier |
| Country | Customer country |

In [ ]:
print("Shape:", df.shape)
df.info()
display(df.isna().sum().rename("missing_rows").to_frame())
display(df.describe(include="all"))

## Audit before cleaning

Counts can overlap. Negative quantities may represent returns or adjustments. Do not remove unusual rows until their meaning is understood.

In [ ]:
duplicate_count = int(df.duplicated().sum())
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["CustomerID"] = df["CustomerID"].astype("string")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="raise")
is_cancellation = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
issues = {
    "Rows with any missing value": int(df.isna().any(axis=1).sum()),
    "Missing customer ID": int(df["CustomerID"].isna().sum()),
    "Cancellation rows": int(is_cancellation.sum()),
    "Nonpositive quantities": int((df["Quantity"] <= 0).sum()),
    "Nonpositive prices": int((df["UnitPrice"] <= 0).sum()),
    "Exact duplicate rows": duplicate_count,
}
audit = pd.DataFrame({"issue": issues.keys(), "affected_rows": issues.values()})
audit["decision"] = "Pending review; source data preserved"
display(audit)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

In [ ]:
month_coverage = df.groupby(df["InvoiceDate"].dt.to_period("M")).agg(
    first_observed=("InvoiceDate", "min"),
    last_observed=("InvoiceDate", "max"),
    invoice_lines=("InvoiceNo", "size"),
)
month_coverage["review_note"] = "Review calendar coverage; transaction dates alone do not prove completeness"
display(month_coverage)

## Record decisions before continuing

Update the audit decisions after inspecting affected rows. Review the first and last months for partial coverage. Preserve rows without customer IDs for overall sales; exclude them from customer metrics.

Next: define purchase sales, save processed data, load SQLite, and answer the five business questions. No sales totals or findings have been calculated yet.